In [39]:
%load_ext autoreload
%autoreload 2

print("Autoreload extension loaded. All modules will be reloaded automatically before executing code.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Autoreload extension loaded. All modules will be reloaded automatically before executing code.


In [40]:
%reload_ext autoreload

import logging

from workspace.config import Config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

config = Config()
print("Loaded", config, "\n")


Loaded Config {
    "seed": 42,
    "model_depth": 1,
    "kernel_depth": 16,
    "kernel_size": 3,
    "epochs": 50,
    "batch_size": 64,
    "use_amp": true,
    "learning_rate": 0.0005,
    "weight_decay": 0.0,
    "num_workers": 2,
    "checkpoint_savedir": "data/checkpoints",
    "checkpoint_interval": 1,
    "checkpoint_restore": "last",
    "Torch version:": "2.13.0+cu130",
    "CUDA supported:": true
} 



In [58]:
import torch
from torch import nn, optim, utils

from workspace.model import SimpleMNISTClassifier
from workspace.state import State
from workspace.trainer import Trainer
from workspace.datasets import MNISTDataset

model = SimpleMNISTClassifier(16)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

state = State(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
)


def train_epoch(config: Config, state: State, loader: utils.data.DataLoader):
    state.model.train()
    total_loss = 0.0
    total_examples = 0

    for images, labels in loader:
        images = images.to(config.device)
        labels = labels.to(config.device)

        state.optimizer.zero_grad()
        logits = state.model(images)
        loss = state.criterion(logits, labels)
        loss.backward()
        state.optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    if state.scheduler is not None:
        state.scheduler.step()

    return total_loss / total_examples


def val_epoch(config: Config, state: State, loader: utils.data.DataLoader):
    state.model.eval()
    total_loss = 0.0
    total_examples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(config.device)
            labels = labels.to(config.device)

            logits = state.model(images)
            loss = state.criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size

    return total_loss / total_examples


ds_train = MNISTDataset(config, train=True)
ds_val = MNISTDataset(config, train=False)
print("Dataset loaded successfully.")

trainer = Trainer(
    config=config,
    state=state,
    train_epoch_fn=train_epoch,
    val_epoch_fn=val_epoch,
)

trainer.fit(ds_train.loader, ds_val.loader, epochs=15)

count = 0
for img, lbl in ds_val.loader:
    img = img.to(config.device)
    lbl = lbl.to(config.device)
    prd = model.predict(img)
    print(f"Predicted: {prd}, Actual: {lbl}")

    if count >= 20:
        break


Dataset loaded successfully.


Training:   0%|          | 0/15 [00:00<?, ?epoch/s]

Loading checkpoint from data/checkpoints/last.pt
No checkpoint found at data/checkpoints/last.pt


Training:   7%|▋         | 1/15 [00:02<00:40,  2.86s/epoch, train_loss=0.305, val_loss=0.0856]

Saving checkpoint to data/checkpoints/last.pt
Saving checkpoint to data/checkpoints/best.pt


...

Training:  93%|█████████▎| 14/15 [00:39<00:02,  2.78s/epoch, train_loss=0.0464, val_loss=0.0497]

Saving checkpoint to data/checkpoints/last.pt


Training: 100%|██████████| 15/15 [00:42<00:00,  2.81s/epoch, train_loss=0.0464, val_loss=0.0497]

Saving checkpoint to data/checkpoints/last.pt


Predicted: tensor([7, 1, 5, 7, 0, 5, 9, 2, 7, 3, 5, 4, 6, 6, 0, 2, 7, 3, 3, 4, 9, 7, 9, 0,
        0, 4, 7, 0, 4, 7, 8, 7, 2, 9, 7, 1, 9, 5, 4, 8, 0, 3, 1, 5, 4, 9, 6, 1,
        0, 7, 7, 4, 4, 5, 0, 2, 5, 3, 1, 6, 4, 5, 5, 9], device='cuda:0'), Actual: tensor([7, 1, 5, 7, 0, 5, 9, 2, 7, 3, 5, 4, 6, 6, 0, 2, 7, 3, 3, 4, 9, 7, 9, 0,
        0, 4, 7, 0, 4, 7, 8, 7, 2, 9, 7, 1, 9, 5, 4, 8, 0, 3, 1, 5, 4, 9, 6, 1,
        0, 7, 7, 4, 4, 5, 0, 2, 5, 3, 1, 6, 4, 5, 5, 9], device='cuda:0')
Predicted: tensor([6, 7, 9, 8, 4, 8, 7, 9, 1, 4, 2, 3, 4, 4, 7, 3, 0, 2, 3, 3, 4, 9, 0, 6,
        8, 9, 5, 0, 6, 1, 7, 5, 1, 2, 8, 1, 9, 4, 6, 0, 7, 1, 6, 1, 4, 9, 2, 2,
        0, 4, 6, 1, 9, 4, 0, 6, 2, 9, 2, 9, 3, 3, 4, 0], device='cuda:0'), Actual: tensor([6, 7, 9, 8, 4, 8, 7, 9, 1, 4, 2, 3, 4, 4, 7, 3, 0, 2, 3, 3, 4, 9, 0, 6,
        8, 9, 5, 0, 6, 1, 7, 5, 1, 2, 4, 1, 9, 4, 6, 0, 7, 1, 6, 1, 4, 9, 2, 2,
        0, 4, 6, 1, 9, 4, 0, 6, 2, 9, 2, 9, 3, 3, 4, 0], device='cuda:0')
Predicted: tensor([9, 8,